In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/raw/data.csv",          
    low_memory=False
)

In [3]:
print(df["success_label"].value_counts())
print(df["success_label"].value_counts(normalize=True))

print(df["still_operating"].value_counts())
print(df["still_operating"].value_counts(normalize=True))

print(df["failure_reason_cat"].value_counts())
print(df["failure_reason_cat"].value_counts(normalize=True))

success_label
-1    40397
 1     2318
 0     1878
Name: count, dtype: int64
success_label
-1    0.905905
 1    0.051981
 0    0.042114
Name: proportion, dtype: float64
still_operating
True     39981
False     4612
Name: count, dtype: int64
still_operating
True     0.896576
False    0.103424
Name: proportion, dtype: float64
failure_reason_cat
unknown            42066
competition          406
business_model       243
funding              242
multiple             241
team                 235
acquisition          233
market_fit           209
legal                206
ran_out_of_cash      147
team_issues          146
no_market_fit        141
other                 78
Name: count, dtype: int64
failure_reason_cat
unknown            0.943332
competition        0.009105
business_model     0.005449
funding            0.005427
multiple           0.005404
team               0.005270
acquisition        0.005225
market_fit         0.004687
legal              0.004620
ran_out_of_cash    0.003296
team_i

In [4]:
print(f"Before dropping cols: {df.shape}")

META_COLS = ["product_id", "name", "brand", "launch_date", "shutdown_date", 'founders', 'website_url', 'case_study_url', 'developer_name', 'marketing_channels', "city"]

df_clean = df.drop(columns=META_COLS, errors="ignore")

print(f"After dropping cols: {df_clean.shape}")

Before dropping cols: (44593, 130)
After dropping cols: (44593, 119)


In [5]:
df_clean["success_factors"] = df_clean["success_factors"].fillna("unknown")
print(df["success_factors"].isnull().sum())

df_clean["failure_reason"] = df_clean["failure_reason"].fillna("unknown")
print(df["failure_reason"].isnull().sum())

df_clean["success_reason"] = df_clean["success_reason"].fillna("unknown")
print(df_clean["success_reason"].isnull().sum())

41025
42938
0


In [6]:
df_clean = df_clean.reset_index(drop=True)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_params = {
    'stop_words': 'english',      
    'ngram_range': (1, 2),        
    'min_df': 3,                   
    'max_df': 0.85,     
    }

tfidf_desc = TfidfVectorizer(max_features=100, **tfidf_params)
desc_matrix = tfidf_desc.fit_transform(df_clean['description'])
desc_df = pd.DataFrame(
    desc_matrix.toarray(),
    columns=[f'desc_{i}' for i in range(desc_matrix.shape[1])]
)
print(f"Created {desc_matrix.shape[1]} features")


tfidf_sf = TfidfVectorizer(max_features=50, **tfidf_params)
sf_matrix = tfidf_sf.fit_transform(df_clean['success_factors'])
sf_df = pd.DataFrame(
    sf_matrix.toarray(),
    columns=[f'sf_{i}' for i in range(sf_matrix.shape[1])]
)
print(f"Created {sf_matrix.shape[1]} features")

tfidf_fr = TfidfVectorizer(max_features=30, **tfidf_params)
fr_matrix = tfidf_fr.fit_transform(df_clean['failure_reason'])
fr_df = pd.DataFrame(
    fr_matrix.toarray(),
    columns=[f'fr_{i}' for i in range(fr_matrix.shape[1])]
)
print(f"Created {fr_matrix.shape[1]} features")

tfidf_sr = TfidfVectorizer(max_features=20, **tfidf_params)
sr_matrix = tfidf_sr.fit_transform(df_clean['success_reason'])
sr_df = pd.DataFrame(
    sr_matrix.toarray(),
    columns=[f'sr_{i}' for i in range(sr_matrix.shape[1])]
)
print(f"Created {sr_matrix.shape[1]} features")

Created 100 features
Created 50 features
Created 30 features
Created 20 features


In [8]:
df_with_text = pd.concat([df_clean, desc_df, sf_df, fr_df, sr_df], axis=1)

df_with_text = df_with_text.drop(
    columns=['description', 'success_factors', 'failure_reason', 'success_reason']
)

print(f"\nBefore: {df_clean.shape}")
print(f"After:  {df_with_text.shape}")
print(f"Added {df_with_text.shape[1] - df_clean.shape[1]} TF-IDF features")


Before: (44593, 119)
After:  (44593, 315)
Added 196 TF-IDF features


In [9]:
print(f"NaN count: {df_with_text.isna().sum().sum()}")

NaN count: 139778


In [10]:
CAT_COLS_TO_FILL = ['category_type', 'product_type', 'country', 
                    'exit_status', 'business_model']

for col in CAT_COLS_TO_FILL:
    if col in df_with_text.columns:
        before = df_with_text[col].isnull().sum()
        df_with_text[col] = df_with_text[col].fillna('unknown')
        print(f"{col}: filled {before} NaN values")

if 'survival_months' in df_with_text.columns:
    before = df_with_text['survival_months'].isnull().sum()
    # median_survival = df_with_text['survival_months'].median()
    df_with_text['survival_months'] = df_with_text['survival_months'].fillna(-1)
    print(f"survival_months: filled {before}")


print(f"\nTotal NaN after filling: {df_with_text.isna().sum().sum()}")


category_type: filled 2958 NaN values
product_type: filled 8374 NaN values
country: filled 32819 NaN values
exit_status: filled 42672 NaN values
business_model: filled 40873 NaN values
survival_months: filled 12082

Total NaN after filling: 0


In [12]:
ONE_HOT_COLS = [
    'main_category',         
    'product_type',          
    'data_source',           
    'exit_status',          
    'failure_reason_cat',    
    'success_reason_cat',  
    'business_model'         
]

one_hot_cols = [col for col in ONE_HOT_COLS if col in df_with_text.columns]

print(f"One-hot encoding {len(one_hot_cols)} columns:")
for col in one_hot_cols:
    n_unique = df_with_text[col].nunique()
    print(f"  - {col}: {n_unique} unique values")

df_encoded = pd.get_dummies(
    df_with_text,
    columns=one_hot_cols,
    drop_first=True,  
    dtype=int
)

print(f"\nBefore one-hot: {df_with_text.shape}")
print(f"After one-hot:  {df_encoded.shape}")
print(f"Added {df_encoded.shape[1] - df_with_text.shape[1]} one-hot features")


One-hot encoding 7 columns:
  - main_category: 33 unique values
  - product_type: 9 unique values
  - data_source: 8 unique values
  - exit_status: 10 unique values
  - failure_reason_cat: 13 unique values
  - success_reason_cat: 3 unique values
  - business_model: 6 unique values

Before one-hot: (44593, 315)
After one-hot:  (44593, 383)
Added 68 one-hot features


In [13]:
from sklearn.preprocessing import LabelEncoder

LABEL_ENCODE_COLS = ['category_type', 'country']
label_cols = [col for col in LABEL_ENCODE_COLS if col in df_encoded.columns]

if len(label_cols) > 0:
    for col in label_cols:
        n_unique = df_encoded[col].nunique()
        print(f"  - {col}: {n_unique} unique values")
        le = LabelEncoder()
        df_encoded[f'{col}_encoded'] = le.fit_transform(df_encoded[col])
        df_encoded = df_encoded.drop(columns=[col])
        
        print(f"Encoded to numeric (0-{df_encoded[f'{col}_encoded'].max()})")
else:
    print("No high-cardinality columns found")

print(f"\nFinal shape after encoding: {df_encoded.shape}")


  - category_type: 315 unique values
Encoded to numeric (0-314)
  - country: 72 unique values
Encoded to numeric (0-71)

Final shape after encoding: (44593, 383)


In [26]:
remaining_objects = df_encoded.select_dtypes(include=['object']).columns.tolist()
if len(remaining_objects) > 0:
    print(f"\n WARNING: {len(remaining_objects)} object columns remaining:")
    for col in remaining_objects:
        print(f"  - {col}: {df_encoded[col].nunique()} unique")
else:
    print("\n No object columns remaining!")

total_nan = df_encoded.isna().sum().sum()
print(f"\n Total NaN values: {total_nan}")
if total_nan > 0:
    nan_cols = df_encoded.isna().sum()
    nan_cols = nan_cols[nan_cols > 0]
    print(f"   Columns with NaN ({len(nan_cols)}):")
    print(nan_cols)

print("\n Column types:")
print(df_encoded.dtypes.value_counts())


print(f"Orginal: {df.shape[1]} | After Drop: {df_clean.shape[1]} | Text Embed: {df_with_text.shape[1]-df_clean.shape[1]} | Encoding: {df_encoded.shape[1]-df_with_text.shape[1]}")
print(f"Consolidated: {(df_clean.shape[1])+(df_with_text.shape[1]-df_clean.shape[1]) + (df_encoded.shape[1]-df_with_text.shape[1])}")
print(f"Rows: {df_encoded.shape[0]}")
print(f"Features: {df_encoded.shape[1]}")
print(f"Final dataset shape: {df_encoded.shape}")


 No object columns remaining!

 Total NaN values: 0

 Column types:
float64    259
int64       97
bool        27
Name: count, dtype: int64
Orginal: 130 | After Drop: 119 | Text Embed: 196 | Encoding: 68
Consolidated: 383
Rows: 44593
Features: 383
Final dataset shape: (44593, 383)


In [28]:
df_clean.to_csv('../data/processed/without_meta/df_without_meta.csv', index=False)
print(f"df_clean Saved!")
df_encoded.to_csv('../data/processed/after_encoding/df_encoded.csv', index=False)
print(f"df_encoded Saved!")

df_clean Saved!
df_encoded Saved!
